# UAV GPS Localization Pipeline
## ICMTC 2026 — Fixed Wing Challenge (UAVC-9)

**Mission:** Convert a YOLO-detected flag pixel in a UAV video frame → GPS coordinate.  
**Scoring:** ≤20 m → 15 pts · 20–30 m → 12 pts · 30–40 m → 9 pts · … · >60 m → 0 pts.

---
### Notebook Structure
| # | Section | Purpose |
|---|---|---|
| 1 | Imports & Configuration | All parameters in one place |
| 2 | Pipeline Mathematics | Full equation walkthrough |
| 3 | Coordinate Systems | The four frames and how they relate |
| 4 | Naive Baseline | Current code — what it does and why it fails |
| 5 | Full Pipeline | Correct ray–plane + IMU rotation |
| 6 | Multi-Frame Aggregation | The biggest accuracy win |
| 7 | Unit Tests | Synthetic ground-truth verification |
| 8 | Error Budget | Sensitivity analysis |
| 9 | Limitations | What cannot be fixed in software |
| 10 | Pre-Flight Checklist | Competition-day runbook |


## Section 1 — Imports & Configuration

All tunable parameters live here. **Edit this cell before each flight session.**

In [3]:
import numpy as np
import math
import random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

# ─────────────────────────────────────────────────────────────────────────
# CAMERA INTRINSICS
# Replace with values from camera_params.yaml after GoPro calibration.
# These MUST be recomputed if GoPro mode or firmware changes.
# ─────────────────────────────────────────────────────────────────────────
@dataclass
class CameraIntrinsics:
    """
    Pinhole camera model.

    Intrinsic matrix K:
        K = [[fx,  0, cx],
             [ 0, fy, cy],
             [ 0,  0,  1]]

    fx, fy  – focal lengths in pixels  (how 'zoomed in' the lens is)
    cx, cy  – principal point          (optical centre, usually ≈ image centre)
    dist    – distortion coefficients  (k1,k2,p1,p2,k3) from cv2.calibrateCamera
    """
    fx: float = 1500.0   # GoPro Linear @ 1080p — REPLACE with your calibration
    fy: float = 1500.0
    cx: float = 960.0    # image_width  / 2
    cy: float = 540.0    # image_height / 2
    dist: np.ndarray = field(
        default_factory=lambda: np.array([0.0, 0.0, 0.0, 0.0, 0.0]))

    @property
    def K(self) -> np.ndarray:
        return np.array([[self.fx,      0, self.cx],
                         [     0, self.fy, self.cy],
                         [     0,       0,       1]], dtype=float)

    def pixels_per_meter(self, altitude_m: float) -> float:
        """How many pixels correspond to 1 metre on the ground at this altitude."""
        return self.fx / altitude_m

    def flag_size_pixels(self, altitude_m: float,
                         flag_w_m: float = 1.0, flag_h_m: float = 2.0):
        """Size of a 1×2 m flag in pixels at this altitude."""
        s = self.pixels_per_meter(altitude_m)
        return s * flag_w_m, s * flag_h_m


# ─────────────────────────────────────────────────────────────────────────
# CAMERA MOUNT
# R_MOUNT maps camera axes to UAV body axes.
# Convention used here:
#   Camera +X  → body +Y  (right in image = right wing when heading forward)
#   Camera +Y  → body +X  (down  in image = forward / nose direction)
#   Camera +Z  → body +Z  (into scene = down for nadir camera)
# If your mount is perfectly aligned this way, R_MOUNT is as below.
# If your camera is rotated 90° around the body Z-axis, encode that here.
# ─────────────────────────────────────────────────────────────────────────
# Correct nadir-camera mount rotation.
# Camera axes to UAV body (NED) frame:
#   cam +X (image right)  → body +Y (East / right wing)
#   cam +Y (image down)   → body -X (South = -North: image down = behind aircraft)
#   cam +Z (into scene)   → body +Z (Down)
#
# Mnemonic: image bottom points toward the TAIL, not the nose.
# A flag that is NORTH of the UAV appears in the UPPER half of the image (v < cy → y_n < 0).
# y_n < 0 must give positive North offset, so cam+Y → body -X.
R_MOUNT = np.array([[ 0, -1, 0],
                    [ 1,  0, 0],
                    [ 0,  0, 1]], dtype=float)

# ─────────────────────────────────────────────────────────────────────────
# GEODESY CONSTANTS (Cairo, Egypt)
# ─────────────────────────────────────────────────────────────────────────
LAT_REF_DEG             = 30.0       # reference latitude for longitude scaling
METERS_PER_DEG_LAT      = 111_320.0  # roughly constant globally
METERS_PER_DEG_LON      = 111_320.0 * math.cos(math.radians(LAT_REF_DEG))  # ≈ 96 378
MAG_DECLINATION_DEG     = 4.0        # Cairo East declination: add to mag heading for true

cam = CameraIntrinsics()
print("   Configuration loaded")
print(f"   Pixels/metre @ 80 m AGL : {cam.pixels_per_meter(80):.1f} px/m")
w, h = cam.flag_size_pixels(80)
print(f"   Flag (1×2 m) at 80 m    : {w:.0f} × {h:.0f} px")
print(f"   1° longitude @ Cairo     : {METERS_PER_DEG_LON:.0f} m")


   Configuration loaded
   Pixels/metre @ 80 m AGL : 18.8 px/m
   Flag (1×2 m) at 80 m    : 19 × 38 px
   1° longitude @ Cairo     : 96406 m


## Section 2 — Pipeline Mathematics

$$
\underbrace{(u_{box},\,v_{box})}_{\text{pixel}} 
\;\xrightarrow{\text{undistort}}\; 
\underbrace{(u',v')}_{\text{ideal pixel}} 
\;\xrightarrow{K^{-1}}\; 
\underbrace{\mathbf{r}_{cam}}_{\text{camera ray}} 
\;\xrightarrow{R_{world}}\; 
\underbrace{\mathbf{r}_{NED}}_{\text{world ray}} 
\;\xrightarrow{t=h/r_z}\; 
\underbrace{(E,\,N)}_{\text{metres}} 
\;\xrightarrow{+\text{UAV GPS}}\; 
\underbrace{(lat_{flag},\,lon_{flag})}_{\text{answer}}
$$

---
### Step 1 — YOLO Bounding Box Center

$$
u_{box} = \frac{x_1 + x_2}{2}, \qquad v_{box} = \frac{y_1 + y_2}{2}
$$

**Why center?** A 1×2 m flag at 80 m covers ≈19×38 px. The geometric center is the best single-point representative. Averaging over multiple frames reduces jitter further.

---
### Step 2 — Undistort the Single Pixel

The lens bends straight lines (especially toward the edges). We must undo this before any geometry.

$$
(u', v') = \text{undistortPoints}\bigl(u_{box},\, v_{box},\; K,\; \mathbf{d}\bigr)
$$

**Key insight:** We undistort **only the one center pixel**, not the whole image. This is ~1000× cheaper and avoids resampling artifacts.

**Why it matters:** An uncalibrated GoPro at the image edge can have 5–10 % radial distortion. At 80 m altitude that moves a flag **5+ metres** on the ground.

---
### Step 3 — Pixel → Normalised Camera Ray

Convert from pixels to a unit-less direction vector using $K$:

$$
x_n = \frac{u' - c_x}{f_x}, \qquad y_n = \frac{v' - c_y}{f_y}
$$

The camera-frame ray is:
$$
\mathbf{r}_{cam} = \begin{bmatrix} x_n \\ y_n \\ 1 \end{bmatrix}
$$

**Intuition:** $x_n = 0.1$ means the flag is 10 % of a focal length to the right.  
$x_n = 0$ means the flag is directly below the camera.

---
### Step 4 — Rotate Ray into World Frame (NED)

This is the step **missing from the naive baseline**. Build a rotation matrix from UAV attitude:

$$
R_{world} = R_z(\psi) \cdot R_y(\theta) \cdot R_x(\phi) \cdot R_{mount}
$$

where $\phi$ = roll, $\theta$ = pitch, $\psi$ = yaw (all from IMU/autopilot).

$$
\mathbf{r}_{NED} = R_{world} \cdot \mathbf{r}_{cam}
$$

**Why yaw-only is wrong:** A 2° pitch (nose-up in wind) causes a ~2.8 m ground error at 80 m that yaw rotation cannot fix.

---
### Step 5 — Ray–Ground Intersection

The UAV is at altitude $h$ above flat ground. The NED ray hits the ground plane ($z_{down} = h$) at:

$$
t = \frac{h}{r_z^{NED}}
$$

$$
\text{East offset} \;=\; t \cdot r_y^{NED}, \qquad \text{North offset} \;=\; t \cdot r_x^{NED}
$$

**Special case — nadir, zero attitude:** Collapses to $E = x_n \cdot h$, $N = -y_n \cdot h$, recovering the naive baseline exactly.

**Guard:** Reject any detection where $r_z^{NED} < 0.17$ (ray more than ~80° from vertical — geometry explodes).

---
### Step 6 — ENU Offset → GPS

$$
\Delta lat = \frac{N}{111{,}320}, \qquad \Delta lon = \frac{E}{111{,}320 \cdot \cos(lat_{UAV})}
$$

$$
lat_{flag} = lat_{UAV} + \Delta lat, \qquad lon_{flag} = lon_{UAV} + \Delta lon
$$

**Cairo specifics:** $\cos(30°) \approx 0.866$, so 1° of longitude ≈ 96,378 m (not 111,320 m).


## Section 3 — Coordinate Systems

There are **four** frames. Confusing two of them is the #1 source of "we are 40 m off and don't know why" bugs.

| Frame | Origin | +X | +Y | +Z | Units |
|---|---|---|---|---|---|
| **Image** | top-left corner | right | **down** | — | pixels |
| **Camera** | optical center | right (image +u) | down (image +v) | out of lens | scaled |
| **Body/NED** | UAV | North (forward) | East (right wing) | **Down** | metres |
| **GPS/WGS84** | Earth center | — | — | — | degrees |

### Y-Axis Trap
Image $+v$ points **down** (standard OpenCV convention, origin at top-left).  
World North points **up** on a map.  
The naive baseline handles this with `Y = −y_n · h`.  
The full pipeline handles it automatically through $R_{mount}$ — **do not apply the sign flip twice**.

### Camera → Body Mount Convention (this code)
| Camera axis | Body (NED) axis | Physical meaning |
|---|---|---|
| $+X$ (image right) | $+Y$ (East/right-wing) | Pixel to the right → flag to the right of aircraft |
| $+Y$ (image down) | $+X$ (North/forward) | Pixel below center → flag in front of aircraft |
| $+Z$ (into scene) | $+Z$ (Down) | Ray goes down toward ground |

If your camera is physically rotated differently on the mount, encode that in `R_MOUNT`.


## Section 4 — Naive Baseline (Current Code)

This is the current simplified pipeline. Correct **only** when the UAV is perfectly level and heading North.

**Known bugs:**
1. ❌ No heading rotation → flags reported in the wrong direction when UAV isn't heading North
2. ❌ No pitch/roll compensation → systematic bias in any wind
3. ❌ No undistortion → edge-of-frame flags off by 5–15 m


In [ ]:
def naive_localize(
    u_box: float, v_box: float,
    fx: float, fy: float, cx: float, cy: float,
    altitude_m: float,
    lat_uav: float, lon_uav: float,
) -> Tuple[float, float]:
    """
    NAIVE baseline (current state).

    Implicit assumptions — ALL must hold for this to be accurate:
    • Camera optical axis is exactly vertical (nadir)
    • UAV heading = 0° (North): image +u → East, image +v → South
    • No distortion (raw pixel used directly)
    • altitude_m is true AGL

    Parameters
    ----------
    u_box, v_box  : bbox center in raw image pixels
    fx, fy        : focal lengths from calibration
    cx, cy        : principal point from calibration
    altitude_m    : UAV height above ground (metres)
    lat_uav       : UAV latitude  (degrees)
    lon_uav       : UAV longitude (degrees)
    """
    # Offset from image centre (pixels)
    dx = u_box - cx
    dy = v_box - cy

    # Normalise by focal length → camera-frame direction
    x_n = dx / fx
    y_n = dy / fy

    # Scale by altitude → ground offset  [BUG: assumes heading = North]
    X_east  =  x_n * altitude_m
    Y_north = -y_n * altitude_m   # sign flip: image +v is South

    # GPS degrees
    delta_lat = Y_north / 111_000.0
    delta_lon = X_east  / (111_000.0 * math.cos(math.radians(lat_uav)))
    return lat_uav + delta_lat, lon_uav + delta_lon


# ── Quick demo ──────────────────────────────────────────────────────────────
cam_demo = CameraIntrinsics()
UAV_LAT, UAV_LON = 30.0500, 31.2500

# Flag directly below the drone (should return exact UAV position)
lat_f, lon_f = naive_localize(cam_demo.cx, cam_demo.cy,
                               cam_demo.fx, cam_demo.fy,
                               cam_demo.cx, cam_demo.cy,
                               80.0, UAV_LAT, UAV_LON)
print(f"Flag directly below: error = {abs(lat_f-UAV_LAT)*111320:.3f} m lat, "
      f"{abs(lon_f-UAV_LON)*METERS_PER_DEG_LON:.3f} m lon")

# Flag 20 m East, UAV heading North — naive works here
px_shift = 20.0 * cam_demo.fx / 80.0
lat_f2, lon_f2 = naive_localize(cam_demo.cx + px_shift, cam_demo.cy,
                                 cam_demo.fx, cam_demo.fy,
                                 cam_demo.cx, cam_demo.cy,
                                 80.0, UAV_LAT, UAV_LON)
print(f"Flag 20 m East, heading N: East={( lon_f2-UAV_LON)*METERS_PER_DEG_LON:.2f} m  "
      f"North={(lat_f2-UAV_LAT)*METERS_PER_DEG_LAT:.2f} m  (correct = 20 m East)")

# BUG: Same pixel, but UAV heading East (yaw=90°) — naive still says East; correct is South
print("\n⚠️  HEADING BUG DEMO")
print("   Pixel is 375 px to the right of image centre.")
print("   Heading North → flag is East  (naive: ✅ correct by coincidence)")
print("   Heading East  → flag is South (naive: ❌ still says East — WRONG direction!)")


## Section 5 — Full Pipeline (Correct Implementation)

With undistortion, full 3-axis attitude rotation, and ray–ground intersection.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Elementary rotation matrices (right-hand rule, radians)
# ─────────────────────────────────────────────────────────────

def rotation_x(angle_rad: float) -> np.ndarray:
    """Rotation around X-axis (roll). Positive = right wing down."""
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    return np.array([[1, 0,  0],
                     [0, c, -s],
                     [0, s,  c]])

def rotation_y(angle_rad: float) -> np.ndarray:
    """Rotation around Y-axis (pitch). Positive = nose up."""
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    return np.array([[ c, 0, s],
                     [ 0, 1, 0],
                     [-s, 0, c]])

def rotation_z(angle_rad: float) -> np.ndarray:
    """Rotation around Z-axis (yaw/heading). Positive = clockwise from North (NED)."""
    c, s = math.cos(angle_rad), math.sin(angle_rad)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def build_world_rotation(roll_deg: float, pitch_deg: float, yaw_deg: float,
                          R_mount: np.ndarray = R_MOUNT) -> np.ndarray:
    """
    Build the camera-to-NED rotation matrix.

    Convention  : ZYX intrinsic Euler angles (autopilot standard).
    Frame out   : NED — +X North, +Y East, +Z Down.

    The matrix satisfies:
        r_NED = R_world @ r_cam

    Parameters
    ----------
    roll_deg  : right-wing-down positive
    pitch_deg : nose-up positive
    yaw_deg   : clockwise from North, 0–360°
    R_mount   : camera-to-body rotation  (see Section 3 and R_MOUNT above)
    """
    r = math.radians(roll_deg)
    p = math.radians(pitch_deg)
    y = math.radians(yaw_deg)
    # ZYX Euler in body/NED frame, then apply mount rotation
    return rotation_z(y) @ rotation_y(p) @ rotation_x(r) @ R_mount


# ─────────────────────────────────────────────────────────────
# Single-pixel undistortion (pure NumPy, no OpenCV required)
# For production use: cv2.undistortPoints(pts, K, dist, P=K)
# ─────────────────────────────────────────────────────────────

def undistort_pixel(u: float, v: float,
                    K: np.ndarray, dist: np.ndarray) -> Tuple[float, float]:
    """
    Undo Brown–Conrady (pinhole-radial) distortion for a single pixel.

    Uses 5 Newton iterations — accurate to < 0.01 px for |k1|,|k2| < 0.3.
    For fisheye (RPi camera), use cv2.fisheye.undistortPoints instead.

    Parameters
    ----------
    u, v  : raw (distorted) pixel coordinates
    K     : 3×3 intrinsic matrix
    dist  : distortion coefficients [k1, k2, p1, p2, k3]
    """
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    k1, k2, p1, p2 = dist[0], dist[1], dist[2], dist[3]
    k3 = dist[4] if len(dist) > 4 else 0.0

    xd, yd = (u - cx) / fx, (v - cy) / fy   # distorted normalised

    x, y = xd, yd
    for _ in range(5):
        r2  = x*x + y*y
        radial = 1 + k1*r2 + k2*r2**2 + k3*r2**3
        dx = 2*p1*x*y + p2*(r2 + 2*x*x)
        dy = p1*(r2 + 2*y*y) + 2*p2*x*y
        x = (xd - dx) / radial
        y = (yd - dy) / radial

    return x * fx + cx, y * fy + cy


# ─────────────────────────────────────────────────────────────
# MAIN LOCALISATION FUNCTION
# ─────────────────────────────────────────────────────────────

def localize(
    u_box: float, v_box: float,
    cam: CameraIntrinsics,
    altitude_m: float,
    lat_uav: float, lon_uav: float,
    roll_deg:  float = 0.0,
    pitch_deg: float = 0.0,
    yaw_deg:   float = 0.0,
    R_mount: np.ndarray = R_MOUNT,
    min_rz: float = 0.17,       # reject rays > ~80° from vertical
) -> Optional[Tuple[float, float]]:
    """
    Full pipeline: pixel coordinate → GPS coordinate.

    Parameters
    ----------
    u_box, v_box : YOLO bounding-box center (raw image pixels)
    cam          : CameraIntrinsics (K matrix + distortion)
    altitude_m   : UAV height above ground level (AGL), metres
    lat_uav      : UAV latitude  at this frame's capture time (degrees)
    lon_uav      : UAV longitude at this frame's capture time (degrees)
    roll_deg     : UAV roll  — right-wing-down positive
    pitch_deg    : UAV pitch — nose-up positive
    yaw_deg      : UAV heading, clockwise from North, 0–360°
    R_mount      : Camera-to-body rotation matrix (default = Section 3 convention)
    min_rz       : Minimum allowed NED down-component (geometry guard)

    Returns
    -------
    (lat_flag, lon_flag) in degrees, or None if geometry is invalid.
    """
    # ── Step 1: Undistort the bbox centre pixel ──────────────
    u_u, v_u = undistort_pixel(u_box, v_box, cam.K, cam.dist)

    # ── Step 2: Pixel → normalised camera ray ────────────────
    x_n = (u_u - cam.cx) / cam.fx
    y_n = (v_u - cam.cy) / cam.fy
    r_cam = np.array([x_n, y_n, 1.0])   # camera frame

    # ── Step 3: Rotate ray into world frame (NED) ─────────────
    R_world = build_world_rotation(roll_deg, pitch_deg, yaw_deg, R_mount)
    r_ned = R_world @ r_cam
    # r_ned = [r_North, r_East, r_Down]

    # ── Step 4: Geometry guard ───────────────────────────────
    # r_ned[2] is the down component. Must be positive (ray heading toward ground).
    if r_ned[2] < min_rz:
        # Ray is nearly horizontal or pointing upward → intersection invalid
        return None

    # ── Step 5: Ray–ground intersection ──────────────────────
    t = altitude_m / r_ned[2]   # distance along ray to ground plane
    North_m = t * r_ned[0]       # North offset (metres)
    East_m  = t * r_ned[1]       # East  offset (metres)

    # ── Step 6: Offset → GPS ─────────────────────────────────
    delta_lat = North_m / METERS_PER_DEG_LAT
    delta_lon = East_m  / METERS_PER_DEG_LON

    return lat_uav + delta_lat, lon_uav + delta_lon


print("✅ Full pipeline defined: rotation_x/y/z, build_world_rotation, undistort_pixel, localize")


## Section 6 — Multi-Frame Aggregation

**Why this is the biggest single accuracy win:**

Per-frame GPS noise is dominated by UAV GPS (~3–5 m), not by detection noise.  
Averaging $N$ independent frames reduces the standard deviation:

$$
\sigma_{final} = \frac{\sigma_{1}}{\sqrt{N}}
$$

| N frames | Expected $\sigma$ (from $\sigma_1 = 5$ m) |
|---|---|
| 1  | 5.0 m |
| 5  | 2.2 m |
| 10 | 1.6 m |
| 20 | 1.1 m |

We use the **median** (not mean) for outlier robustness — a single bad frame where YOLO fires on a shadow won't ruin the estimate.


In [ ]:
def localize_flag(
    detections: List[dict],
    cam: CameraIntrinsics,
    min_confidence: float = 0.4,
) -> Optional[Tuple[float, float]]:
    """
    Aggregate per-frame localizations into a robust GPS estimate.

    Parameters
    ----------
    detections : list of dicts, each containing:
        {
          'u'         : float,   # bbox centre x (pixels)
          'v'         : float,   # bbox centre y (pixels)
          'altitude'  : float,   # AGL at frame time (metres)
          'lat_uav'   : float,   # UAV latitude  at frame time
          'lon_uav'   : float,   # UAV longitude at frame time
          'roll'      : float,   # degrees  (optional, default 0)
          'pitch'     : float,   # degrees  (optional, default 0)
          'yaw'       : float,   # degrees, clockwise from North (optional, default 0)
          'confidence': float,   # YOLO detection confidence 0–1
        }
    cam            : CameraIntrinsics
    min_confidence : detections below this threshold are ignored

    Returns
    -------
    (lat_flag, lon_flag) median estimate, or None if no valid frames.
    """
    estimates = []
    for d in detections:
        if d.get('confidence', 1.0) < min_confidence:
            continue
        result = localize(
            u_box      = d['u'],
            v_box      = d['v'],
            cam        = cam,
            altitude_m = d['altitude'],
            lat_uav    = d['lat_uav'],
            lon_uav    = d['lon_uav'],
            roll_deg   = d.get('roll',  0.0),
            pitch_deg  = d.get('pitch', 0.0),
            yaw_deg    = d.get('yaw',   0.0),
        )
        if result is not None:
            estimates.append(result)

    if not estimates:
        return None

    arr = np.array(estimates)   # shape (N, 2)
    return float(np.median(arr[:, 0])), float(np.median(arr[:, 1]))


def haversine_meters(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Great-circle distance between two GPS points (metres).
    Accurate to ~0.5 % for distances < 100 km.
    Used to evaluate localisation error vs ground truth.
    """
    R = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def score_localization(est_lat: float, est_lon: float,
                       true_lat: float, true_lon: float) -> Tuple[int, float]:
    """Return (competition_points, distance_m) for ICMTC 2026 scoring bins."""
    d = haversine_meters(est_lat, est_lon, true_lat, true_lon)
    pts = 15 if d <= 20 else (12 if d <= 30 else (9 if d <= 40 else
          (6 if d <= 50 else (3 if d <= 60 else 0))))
    return pts, d


print("✅ Multi-frame aggregation and scoring functions defined.")


## Section 7 — Unit Tests

Synthetic ground-truth tests. **All must pass before processing real data.**

In [ ]:
cam = CameraIntrinsics()
UAV_LAT, UAV_LON = 30.0500, 31.2500
PASS, FAIL = "✅ PASS", "❌ FAIL"
TOL = 0.10   # metres numerical tolerance

def check(label, estimated, true_lat, true_lon, tol=TOL):
    if estimated is None:
        print(f"  {FAIL}  {label}: pipeline returned None")
        return
    d = haversine_meters(estimated[0], estimated[1], true_lat, true_lon)
    ok = d <= tol
    print(f"  {'✅ PASS' if ok else '❌ FAIL'}  {label}:  error = {d:.4f} m")

print("═"*60)
print("GROUP 1 — Flag directly below drone (all attitudes)")
print("═"*60)
for yaw in [0, 45, 90, 180, 270]:
    r = localize(cam.cx, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=yaw)
    check(f"Below, yaw={yaw:3d}°", r, UAV_LAT, UAV_LON)

print()
print("═"*60)
print("GROUP 2 — Heading North (yaw=0°), known offsets")
print("═"*60)
# 20 m East: pixel shifts right by 20*(fx/h) pixels
px = 20.0 * cam.fx / 80.0
r = localize(cam.cx + px, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=0)
check("20 m East, yaw=0°", r, UAV_LAT, UAV_LON + 20/METERS_PER_DEG_LON)

# 15 m North: pixel shifts UP in image (lower v) by 15*(fy/h)
py = -15.0 * cam.fy / 80.0   # Flag North of UAV → appears in UPPER image (v < cy) → py negative
# R_MOUNT: cam+Y(image-down) → body-X(South), so image-up(y_n<0) → North ✅
r = localize(cam.cx, cam.cy + py, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=0)
check("15 m North, yaw=0°", r, UAV_LAT + 15/METERS_PER_DEG_LAT, UAV_LON)

print()
print("═"*60)
print("GROUP 3 — Heading rotation (critical test)")
print("═"*60)
px = 20.0 * cam.fx / 80.0
# UAV heading East (90°): flag to image-right = South of UAV
r = localize(cam.cx + px, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=90)
check("20m right-in-image, yaw=90°E → 20m South",
      r, UAV_LAT - 20/METERS_PER_DEG_LAT, UAV_LON, tol=0.5)

# UAV heading South (180°): flag to image-right = West of UAV
r = localize(cam.cx + px, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=180)
check("20m right-in-image, yaw=180°S → 20m West",
      r, UAV_LAT, UAV_LON - 20/METERS_PER_DEG_LON, tol=0.5)

# UAV heading West (270°): flag to image-right = North of UAV
r = localize(cam.cx + px, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, yaw_deg=270)
check("20m right-in-image, yaw=270°W → 20m North",
      r, UAV_LAT + 20/METERS_PER_DEG_LAT, UAV_LON, tol=0.5)

print()
print("═"*60)
print("GROUP 4 — Altitude linear scaling")
print("═"*60)
px = 10.0 * cam.fx / 80.0
r80  = localize(cam.cx + px, cam.cy, cam,  80.0, UAV_LAT, UAV_LON)
r160 = localize(cam.cx + px, cam.cy, cam, 160.0, UAV_LAT, UAV_LON)
if r80 and r160:
    e80  = (r80[1]  - UAV_LON) * METERS_PER_DEG_LON
    e160 = (r160[1] - UAV_LON) * METERS_PER_DEG_LON
    ratio = e160 / e80
    ok = abs(ratio - 2.0) < 0.001
    print(f"  {'✅ PASS' if ok else '❌ FAIL'}  Double altitude → offset ratio = {ratio:.4f} (expected 2.0)")

print()
print("═"*60)
print("GROUP 5 — Geometry guard")
print("═"*60)
r = localize(cam.cx, cam.cy, cam, 80.0, UAV_LAT, UAV_LON, pitch_deg=85)
ok = r is None
print(f"  {'✅ PASS' if ok else '❌ FAIL'}  85° pitch → returns None  (got: {r})")

print()
print("═"*60)
print("GROUP 6 — Multi-frame averaging reduces noise")
print("═"*60)
random.seed(42)
TRUE_FLAG_LAT = UAV_LAT
TRUE_FLAG_LON = UAV_LON + 20.0 / METERS_PER_DEG_LON
px = 20.0 * cam.fx / 80.0

detections = [{
    'u': cam.cx + px, 'v': cam.cy, 'altitude': 80.0,
    'lat_uav': UAV_LAT + random.gauss(0, 3/METERS_PER_DEG_LAT),
    'lon_uav': UAV_LON + random.gauss(0, 3/METERS_PER_DEG_LON),
    'roll': 0., 'pitch': 0., 'yaw': 0., 'confidence': 0.85,
} for _ in range(10)]

single = localize(detections[0]['u'], detections[0]['v'], cam,
                  80.0, detections[0]['lat_uav'], detections[0]['lon_uav'])
multi  = localize_flag(detections, cam)

e1 = haversine_meters(*single, TRUE_FLAG_LAT, TRUE_FLAG_LON)
eN = haversine_meters(*multi,  TRUE_FLAG_LAT, TRUE_FLAG_LON)
print(f"  Single-frame error : {e1:.2f} m")
print(f"  10-frame median    : {eN:.2f} m")
ok = eN < e1
print(f"  {'✅ PASS' if ok else '⚠️  (noise seed may give atypical result)'}  Multi-frame error < single-frame error")

print()
print("═"*60)
print("GROUP 7 — Confidence threshold filters bad detections")
print("═"*60)
bad_det = {'u': 0., 'v': 0., 'altitude': 80., 'lat_uav': UAV_LAT, 'lon_uav': UAV_LON,
           'roll': 0., 'pitch': 0., 'yaw': 0., 'confidence': 0.2}
good_det = {'u': cam.cx + px, 'v': cam.cy, 'altitude': 80.,
            'lat_uav': UAV_LAT, 'lon_uav': UAV_LON,
            'roll': 0., 'pitch': 0., 'yaw': 0., 'confidence': 0.9}
result = localize_flag([bad_det, good_det], cam)
e = haversine_meters(*result, TRUE_FLAG_LAT, TRUE_FLAG_LON)
ok = e < 0.5
print(f"  {'✅ PASS' if ok else '❌ FAIL'}  Low-confidence frame filtered: error = {e:.4f} m")

print()
print("═"*60)
print("GROUP 8 — Scoring function")
print("═"*60)
for dist_m, expected_pts in [(10,15),(25,12),(35,9),(45,6),(55,3),(70,0)]:
    pts, d = score_localization(UAV_LAT, UAV_LON,
                                UAV_LAT + dist_m/METERS_PER_DEG_LAT, UAV_LON)
    ok = pts == expected_pts
    print(f"  {'✅ PASS' if ok else '❌ FAIL'}  {dist_m:2d} m error → {pts} pts (expected {expected_pts})")


## Section 8 — Error Budget & Sensitivity Analysis

Each error source is varied independently to show its contribution to ground error.

In [ ]:
cam = CameraIntrinsics()
UAV_LAT, UAV_LON = 30.05, 31.25
H = 80.0         # AGL altitude (m)
OFFSET_M = 20.0  # flag is 20 m East of UAV

TRUE_LAT = UAV_LAT
TRUE_LON = UAV_LON + OFFSET_M / METERS_PER_DEG_LON
px = OFFSET_M * cam.fx / H

baseline = localize(cam.cx + px, cam.cy, cam, H, UAV_LAT, UAV_LON)
base_err = haversine_meters(*baseline, TRUE_LAT, TRUE_LON)

def Δ(r): return abs(haversine_meters(*r, TRUE_LAT, TRUE_LON) - base_err)

rows = []

# 1. Pixel jitter
r = localize(cam.cx + px + 3, cam.cy, cam, H, UAV_LAT, UAV_LON)
rows.append(("Bbox centre jitter ±3 px",    Δ(r),         "Negligible"))

# 2. fx calibration error 2%
c2 = CameraIntrinsics(fx=cam.fx*1.02)
r = localize(cam.cx + px, cam.cy, c2, H, UAV_LAT, UAV_LON)
rows.append(("Calibration error fx +2%",    Δ(r),         "Low"))

# 3. Altitude +5 m
r = localize(cam.cx + px, cam.cy, cam, H+5, UAV_LAT, UAV_LON)
rows.append(("Altitude error +5 m (6%)",    Δ(r),         "Medium"))

# 4. 2° pitch WITH IMU compensation
r = localize(cam.cx + px, cam.cy, cam, H, UAV_LAT, UAV_LON, pitch_deg=2.0)
rows.append(("2° pitch — WITH IMU (full pipeline)", Δ(r), "Low ✅"))

# 5. 2° pitch WITHOUT compensation (naive approximation)
tilt_error = H * math.tan(math.radians(2.0))  # metres
rows.append(("2° pitch — WITHOUT IMU (naive baseline)", tilt_error, "HIGH ⚠️"))

# 6. Heading error 2°
r = localize(cam.cx + px, cam.cy, cam, H, UAV_LAT, UAV_LON, yaw_deg=2.0)
rows.append(("Heading error ±2°",            Δ(r),         "Low–Med"))

# 7. UAV GPS noise (irreducible floor)
rows.append(("UAV GPS noise ±3 m CEP",       3.0,          "HIGH ⚠️ (irreducible)"))

# 8. Time sync 200 ms @ 20 m/s UAV speed
rows.append(("Time sync 200 ms @ 20 m/s",   20 * 0.2,     "HIGH ⚠️"))

# 9. GoPro EIS ON
rows.append(("GoPro EIS ON (K shift+crop)", "5–15 m",     "CRITICAL 🚨"))

print(f"Flag at {OFFSET_M} m East, UAV at {H} m AGL")
print("─"*66)
print(f"{'Source':<42} {'Error':>8}    Severity")
print("─"*66)
for name, val, sev in rows:
    if isinstance(val, float):
        print(f"  {name:<40} {val:>6.2f} m    {sev}")
    else:
        print(f"  {name:<40} {val:>8}    {sev}")

numeric = [r[1] for r in rows if isinstance(r[1], float)]
rss = math.sqrt(sum(x**2 for x in numeric))
# With 10-frame avg: GPS noise goes from 3 → 3/sqrt(10)
rss_avg = math.sqrt(sum(x**2 for x in numeric) - 9 + 9/10)
print("─"*66)
print(f"  {'RSS total (excl. EIS)':<40} {rss:>6.2f} m")
print(f"  {'RSS with 10-frame avg':<40} {rss_avg:>6.2f} m")


## Section 9 — Known Limitations

### 9.1 What Cannot Be Fixed in Software

| Limitation | Root Cause | Effect | Mitigation |
|---|---|---|---|
| **UAV GPS noise floor** | Consumer GPS, ~2–5 m CEP | Irreducible ~3 m per frame | Multi-frame averaging is the only cure |
| **Rolling shutter** | CMOS sensor reads rows sequentially | Each row from a slightly different UAV position | High shutter speed; reject frames during turns |
| **GoPro EIS / HyperSmooth** | In-camera digital stabilisation crops and warps | Intrinsics change per frame — uncalibratable | **Disable EIS before every flight** |
| **Magnetic declination** | Cairo: ~4° East | Heading from magnetometer off by 4° → ~1.4 m error at 20 m offset | Add 4° to magnetic yaw, or use GPS course-over-ground |

---
### 9.2 Model Assumptions That Can Break

| Assumption | When it breaks | Consequence |
|---|---|---|
| Flat ground | Terrain relief > 1 m | Altitude error → proportional localisation error |
| Constant altitude during detection burst | UAV climbs/descends during burst | Each frame has different effective $h$ |
| Camera mount truly nadir | Mount not level; vibrations | Systematic bias in all localisations |
| Pinhole model sufficient | GoPro in Wide/SuperView mode | Residual distortion at edges; use Linear mode only |

---
### 9.3 Computational Limitations

The competition video is ~15 minutes. Processing speed vs quality tradeoffs:

| Strategy | Speed | Detection Quality | Recommended? |
|---|---|---|---|
| Every frame (1080p30) | ~5–10 min | Best | Offline post-flight only |
| Frame sampling (1 per 5 frames) | 5× faster | Good (flag visible for seconds) | **Yes — default** |
| SAHI tiled inference | 3–5× slower | Better on tiny flags | Only if baseline misses flags |
| GPU batch inference | 3–4× faster | Same | Yes if GPU available |

**Competition constraint:** No internet, no cloud. All compute is local.

---
### 9.4 Honest Accuracy Expectations

| Scenario | Expected mean error | Notes |
|---|---|---|
| Ideal: calm, good cal, EIS off, 10+ frames | 3–6 m | Achievable |
| Typical: some wind, 5 frames | 8–12 m | Still in 20 m bin mostly |
| Heading rotation missing | 15–40 m | Likely fails scoring |
| EIS left ON | 10–50 m | Unpredictable |
| Single-frame estimate | 5–15 m | High variance |

**Target:** EIS off + proper calibration + heading rotation + 5+ frame averaging → ≥80% of detections in the 20 m bin (full points).


## Section 10 — Pre-Flight Checklist (Localisation-Specific)

Print and physically tick before every flight.

```
CAMERA
[ ] GoPro mode = Linear   (verify on screen, not from memory)
[ ] HyperSmooth / EIS     = OFF
[ ] Horizon Lock          = OFF
[ ] Resolution & FPS match calibration session
[ ] Mode sticker on camera body matches settings menu

CALIBRATION
[ ] camera_params.yaml loaded; mode matches GoPro screen
[ ] Reprojection RMS < 1.0 px  (from last calibration run)
[ ] Sanity check: undistorted grid lines look straight on screen

MOUNT
[ ] Camera mount level — checked with digital level on bench
[ ] Camera-to-body yaw offset documented (target: 0°)

TELEMETRY
[ ] Autopilot telemetry logging enabled
[ ] Barometer ground-pressure calibration set at takeoff point
[ ] Magnetic declination +4° E entered in autopilot (Cairo)
[ ] Frame timestamp ↔ GPS timestamp alignment verified
[ ] Interpolation gap limit set to 200 ms

PIPELINE
[ ] All unit tests in Section 7 pass  (run now: all ✅)
[ ] Heading rotation enabled  (full pipeline, not naive baseline)
[ ] Multi-frame aggregation N ≥ 5 per flag
[ ] YOLO confidence threshold ≥ 0.4

POST-FLIGHT
[ ] Validate at least one known ground point per flight
[ ] Log mean error in error-budget spreadsheet
[ ] Note any unexpected results for next calibration session
```
